
# Programming Assignment 1: Linear Models

**Released:** September 18, 2026

**Due:** October 2, 2026 at 11:59 PM

**Name:**

**Email:**

**Total: 100 points**

**Library policy.** You may not import any library beyond those in the import cell below. Use PyTorch for numerical computations, linear algebra, and implementing the regression models. You may use pandas for loading and inspecting the dataset and Matplotlib for visualization.

Linear Regression and Ridge Regression must be implemented using the closed-form solutions specified in Tasks 2.2 and 2.3. Do not use Gradient Descent (GD), Stochastic Gradient Descent (SGD), `torch.nn`, `torch.optim`, or built-in Linear Regression or Ridge Regression models.

The only permitted scikit-learn functions are `train_test_split` (Task 2.1) and `Lasso` (Task 2.4). Lasso does not have a general closed-form solution, so the library implementation is permitted for that task. You may convert PyTorch tensors to NumPy arrays only when passing data to or retrieving results from these permitted scikit-learn functions.

You may not use scikit-learn for anything else, including cross-validation, polynomial feature construction, or evaluation metrics.

Use `torch.float64` for numerical computations throughout the assignment.

**Answering the written parts.** Several tasks ask you to report specific numbers. Print them from your code and restate them in the markdown cell that follows.

In [ ]:

import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split  # Task 2.1 only
from sklearn.linear_model import Lasso                # Task 2.4 only

torch.manual_seed(42)


## Z Disease Dataset

The Z dataset contains ten baseline variables (age, sex, BMI, average blood pressure, and six blood serum measurements, `s1` through `s6`) and a quantitative measure of disease progression one year after baseline.

It comprises 442 samples, each with 10 input features and a target value representing the quantitative measure of disease Z progression.

Two facts you will need later:

1. All ten input features have already been mean-centered and scaled to unit norm. They are therefore on a comparable scale, so the magnitudes of fitted coefficients can be compared directly against one another.

2. `s1` through `s6` are six different blood serum measurements; `s1` is the first of these. The target is continuous, not a class label.

For this assignment, load the dataset using pandas and convert the feature matrix and target vector to PyTorch tensors with `dtype=torch.float64`.

In [ ]:

# Load the entire dataset from the CSV file
data = pd.read_csv('hw1_dataset.csv')

# Separate the features, target values, and feature names
X = torch.tensor(
    data.drop('target', axis=1).values,
    dtype=torch.float64
)

y = torch.tensor(
    data['target'].values,
    dtype=torch.float64
)

feature_names = data.drop('target', axis=1).columns.tolist()

print(X.shape, y.shape)
print(feature_names)

data.head()

## Question 1: Exploratory Data Analysis and Visualization (20 points)


### Task 1.1 (5 points): Feature Target Relationship

Create a scatter plot to visualize the relationship between average blood pressure (`bp`) and disease Z progression (`target`).

- Calculate the **Pearson correlation coefficient** between `bp` and `target` using PyTorch.
- Based on the calculated correlation coefficient, state whether the relationship is positive or negative.
- Based on both the scatter plot and the calculated correlation, explain in **2-3 sentences** whether `bp` alone appears sufficient for accurately predicting disease Z progression.

**Note:** Do not use a machine learning model for this task.

The Pearson correlation coefficient measures the strength and direction of the linear relationship between two variables. It is defined as

$$
r_{xy} =
\frac{
\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})
}{
\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}
\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}
}
$$

where $\bar{x}$ and $\bar{y}$ are the sample means of $x$ and $y$, respectively.

The coefficient ranges from $-1$ to $1$: values closer to $1$ indicate a stronger positive linear relationship, values closer to $-1$ indicate a stronger negative linear relationship, and values near $0$ indicate little or no linear relationship.

In [ ]:
# Your code here

*Your answer here.*


### Task 1.2 (10 points): Comparing Feature Distributions

Create histograms to visualize the distributions of BMI (`bmi`) and average blood pressure (`bp`). Use **15 bins** for both histograms.

For each feature:

- Calculate and report the **mean** using PyTorch.
- Calculate and report the **standard deviation** using PyTorch.

Specifically, report the standard deviations of `bmi` and `bp`, state whether they are equal or different (up to numerical precision), and report the absolute difference between them.

Then compute and report the standard deviation of each of the ten input features.

- What do you notice about these ten values? Are the standard deviations of all ten input features approximately equal? Report the minimum and maximum standard deviation across the ten features and use these values to support your answer.

Because the standard deviations are approximately equal, they do not distinguish which of these two features has a larger range. Compute the range (maximum minus minimum) of `bmi` and `bp` instead, and report which of the two is larger.

In **1–2 sentences**, state what preprocessing step would produce standard deviations like these, and why that fact will matter when you compare fitted coefficients across features in Question 3.

In [ ]:
# Your code here

*Your answer here.*


### Task 1.3 (5 points): Feature Correlation Analysis

Calculate the **Pearson correlation coefficient** between each of the 10 input features and disease Z progression (`target`) using PyTorch.

Create a bar plot where:

- The x-axis represents the input features.
- The y-axis represents the corresponding correlation coefficient with `target`.

Based on your results, answer the following:

1. Which feature has the largest positive correlation with disease Z progression?
2. Which feature has the largest negative correlation with disease Z progression?
3. Does a large absolute correlation necessarily imply that the feature causes disease Z progression? Explain your answer in **1–2 sentences**.

In [ ]:
# Your code here

*Your answer here.*

## Question 2: Linear, Ridge, and Lasso Regression (25 points)


### Task 2.1 (5 points): Train/Test Split

Split the dataset into training and testing sets using an **80/20 split**.

Use `random_state=42` to ensure reproducibility.

You may use the permitted `sklearn.model_selection.train_test_split` function to split the sample indices, then use those indices to obtain the corresponding PyTorch training and testing tensors.

Keep the feature matrices and target vectors as PyTorch tensors with `dtype=torch.float64` for the remaining tasks.

Report the number of samples in the training set and the test set.

In [ ]:
# Your code here


### Task 2.2 (10 points): Linear Regression

A linear regression model predicts the target as a linear combination of the input features. The model parameters are chosen to minimize the sum of squared errors:

$$
\mathbf{w}^{*}
=
\arg\min_{\mathbf{w}}
\sum_{i=1}^{n}
\left(
y_i-\mathbf{w}^{T}\mathbf{x}^{(i)}
\right)^2
$$

Where:

- $\mathbf{x}^{(i)}$ represents the input features for the $i$-th example.
- $y_i$ represents the target value for the $i$-th example.
- $\mathbf{w}$ represents the model parameters (a.k.a., coefficients).

All models include an **unpenalized intercept**. For the models you implement yourself (Linear Regression here and Ridge Regression in Task 2.3), fit the intercept by appending a column of ones as the **first column** of the feature matrix. The intercept must be excluded from the Ridge penalty.

`Lasso` in Task 2.4 fits its own unpenalized intercept, so pass it the original feature matrix without an additional column of ones.

Fit a **Linear Regression** model using the training data.

**Required fitting method: Closed-form solution using the normal equations**

Compute the Linear Regression parameters using the following closed-form least-squares solution:

$$
\boxed{
\mathbf{w}^{*}
=
\left(
\widetilde{\mathbf{X}}_{\text{train}}^{T}
\widetilde{\mathbf{X}}_{\text{train}}
\right)^{-1}
\widetilde{\mathbf{X}}_{\text{train}}^{T}
\mathbf{y}_{\text{train}}
}
$$

Here, $\widetilde{\mathbf{X}}_{\text{train}}$ is the training feature matrix after appending a column of ones as its first column.

Use PyTorch matrix operations to construct the normal equations and `torch.linalg.solve()` to solve the resulting linear system.

You do not need to calculate the matrix inverse explicitly.

**Do not use GD, SGD, `torch.nn`, `torch.optim`, or a built-in Linear Regression model.**

Using the testing set:

- Generate predictions for the target variable using the fitted parameters.
- Calculate and report the **Mean Squared Error (MSE)** using PyTorch.
- Create a scatter plot with the **actual target values on the x-axis** and the **predicted target values on the y-axis**.
- Add a reference line to the plot showing where a **perfect prediction** would fall. Specifically, draw the line $y=x$ from (minimum target value, minimum target value) to (maximum target value, maximum target value). A point on this line means that the model's predicted value exactly matches the actual target value for that sample. The vertical distance from a point to this line represents the absolute prediction error for that sample.

In [ ]:
# Your code here


### Task 2.3 (5 points): Ridge Regression

Ridge Regression is a regularized version of Linear Regression that adds an L2 penalty to the linear regression objective function:

$$
\mathbf{w}^{*}
=
\arg\min_{\mathbf{w}}
\left[
\sum_{i=1}^{n}
\left(
y_i-\mathbf{w}^{T}\mathbf{x}^{(i)}
\right)^2
+
\lambda\mathbf{w}^{T}\mathbf{P}\mathbf{w}
\right]
$$

where $\lambda$ is the regularization hyperparameter that controls the strength of the L2 penalty.

Here, $\mathbf{P}$ is a diagonal matrix that excludes the intercept from regularization. Since the intercept is the first entry in $\mathbf{w}$, define

$$
\mathbf{P}
=
\operatorname{diag}(0,1,1,\ldots,1).
$$

**Required fitting method: Closed-form Ridge Regression**

Implement Ridge Regression using the following closed-form solution:

$$
\boxed{
\mathbf{w}_{\text{ridge}}^{*}
=
\left(
\widetilde{\mathbf{X}}_{\text{train}}^{T}
\widetilde{\mathbf{X}}_{\text{train}}
+
\lambda\mathbf{P}
\right)^{-1}
\widetilde{\mathbf{X}}_{\text{train}}^{T}
\mathbf{y}_{\text{train}}
}
$$

Here, $\widetilde{\mathbf{X}}_{\text{train}}$ includes a column of ones as its first column.

Use PyTorch matrix operations and `torch.linalg.solve()` to compute the Ridge Regression parameters.

You do not need to calculate the matrix inverse explicitly.

**Do not use GD, SGD, `torch.nn`, `torch.optim`, or a built-in Ridge Regression model.**

Fit a Ridge Regression model using:

$$
\lambda=1
$$

Then:

- Calculate and report the **testing MSE** using PyTorch.
- Report the coefficient associated with each input feature.
- Calculate the **L2 norm** of the coefficient vector for both the Linear Regression model (from Task 2.2) and the Ridge Regression model, excluding the intercept.
- Which model has the smaller coefficient norm?

In [ ]:
# Your code here

*Your answer here.*


### Task 2.4 (5 points): Lasso Regression

Lasso Regression is a regularized version of Linear Regression that adds an L1 penalty to the linear regression objective function:

$$
\mathbf{w}^{*}
=
\arg\min_{\mathbf{w}}
\left[
\frac{1}{2n}
\sum_{i=1}^{n}
\left(
y_i-\mathbf{w}^{T}\mathbf{x}^{(i)}
\right)^2
+
\lambda\|\mathbf{w}\|_1
\right]
$$

The intercept is included in the predictions but excluded from the L1 penalty.

Here, $n$ is the number of training samples. With the objective written this way, you may pass `alpha=lam` directly to `Lasso`; no conversion is needed.

Where $\lambda$ is the regularization hyperparameter that controls the strength of the L1 penalty.

**Required fitting method: Permitted scikit-learn Lasso implementation**

Use `sklearn.linear_model.Lasso` with `alpha=1` and `fit_intercept=True`.

Since Lasso fits its own unpenalized intercept, pass the original training feature matrix without appending a column of ones.

You may convert the PyTorch training and testing tensors to NumPy arrays when passing data to the permitted Lasso implementation.

Use PyTorch for subsequent numerical calculations and evaluation metrics.

Fit a Lasso Regression model using:

$$
\lambda=1
$$

Then:

- Calculate and report the **testing MSE**.
- Report the coefficient associated with each input feature.
- Report the number of coefficients that are exactly zero.
- In **1–2 sentences**, explain what the zero coefficients indicate about the effect of L1 regularization.

In [ ]:
# Your code here

*Your answer here.*

---
## Question 3: Model Analysis and Interpretation (15 points)


### Task 3.1 (5 points): Comparing the Fitted Weights

Produce a grouped bar plot comparing the fitted coefficient values of the three models from Question 2.

Put the ten feature names on the x-axis and the coefficient value $w_j$ on the y-axis, with three bars per feature, one each for the Linear Regression, Ridge ($\lambda=1$), and Lasso ($\lambda=1$) models.

Include a legend identifying the three models and a horizontal line at $y=0$, since coefficients can be negative.

Recall from the dataset description that the ten features were already centered and scaled before release, so the magnitudes of these coefficients are directly comparable across features.

All models include an unpenalized intercept. Linear Regression and Ridge Regression fit the intercept using a column of ones, while Lasso fits its intercept internally. The intercept is not shown in this plot.

The Ridge and Lasso objectives normalize the squared-error term differently, so $\lambda=1$ does not denote the same penalty strength for both. This plot compares the three fitted models, not three equal penalty strengths.

Report:

- The `s1` coefficient of the Linear Regression model. State by how much and in which direction the predicted progression changes for a **one standard deviation** increase in `s1`, with all other features held fixed. (`s1` is the first of the six blood serum measurements.)

- The `bmi` coefficient under all three models, naming which is largest in absolute value.

Then answer in **1–2 sentences**, referring back to your bar plot from Task 1.3:

Does the feature with the largest absolute correlation with the target also receive the largest absolute coefficient in the Linear Regression model?

Name the two features and say what this tells you about reading correlations as if they were model weights.

In [ ]:
# Your code here

*Your answer here.*


### Task 3.2 (5 points): Regularization Paths

Fit Ridge and Lasso across the grid

`lambdas = torch.logspace(-3, 3, 50, dtype=torch.float64)`

refitting on the training set at every value.

For Ridge Regression, use the closed-form solution from Task 2.3 with the corresponding value of $\lambda$.

For Lasso, use the permitted `sklearn.linear_model.Lasso` implementation with `alpha=lam` and `fit_intercept=True`. Convert tensors to NumPy arrays only when required by this implementation.

Produce a figure with two panels side by side. In each panel, plot all ten coefficients against $\log_{10}(\lambda)$, one line per feature, with a legend.

The left panel is Ridge, and the right panel is Lasso.

For this task, treat a coefficient as zero if its absolute value is less than $10^{-8}$.

Report:

- The approximate $\lambda$ at which Lasso first drives at least five of the ten coefficients to zero under this tolerance.
- The three features whose Lasso coefficients survive to the largest $\lambda$.

Then answer in two sentences:

As $\lambda$ grows large, do the Ridge coefficients generally become exactly zero, or do they shrink toward zero? Which of the two methods performs subset selection?

In [ ]:
# Your code here

*Your answer here.*


### Task 3.3 (5 points): Regularization Is Not Scale-Invariant

Make a copy of the training and test feature matrices using PyTorch and multiply the `bmi` column by 1000 in both.

Do not re-standardize or re-scale the `bmi` column after multiplying it.

Refit two models directly on the rescaled matrices:

- Ordinary least squares, using the closed-form solution from Task 2.2.
- Ridge Regression with $\lambda=1.0$, using the closed-form solution from Task 2.3.

Use the same intercept convention as in Question 2: a column of ones as the first column, with the intercept excluded from regularization.

Report, for each of the two models:

- The `bmi` coefficient before and after rescaling.
- The test MSE before and after rescaling.

Then answer in **2–3 sentences**:

Why do the predictions of ordinary least squares stay essentially unchanged under this rescaling while the predictions of Ridge Regression may change?

Refer to the form of the penalty term.

In [ ]:
# Your code here

*Your answer here.*

---
## Question 4: The Bias-Variance Tradeoff (20 points)

### Task 4.1 (10 points): The decomposition

Answer the following in markdown. No code is required. Keep each answer to one or two sentences.

1. Write the decomposition of the expected squared error at a fixed input point $\mathbf{x}_0$ into its three components, name each term, and say which one is irreducible.
2. In the typical bias-variance tradeoff, how do the squared bias and variance terms generally change as model complexity increases? What happens to the irreducible noise term, and why?
3. This decomposition is derived for **squared** error. Does the same clean three-term decomposition hold if you measure error with the absolute deviation $|y - \hat{y}|$ instead?

*Your answer here.*


### Task 4.2 (10 points): Polynomial Regression and Overfitting

To make this experiment reproducible and gradeable, the setup is fully specified. Follow it exactly.

- Use only the `bmi` feature. Ignore the other nine.

- For each degree $d\in\{1,2,\ldots,8\}$, build the data matrix

$$
\widetilde{\mathbf{X}}_d
=
[1,x,x^2,\ldots,x^d].
$$

There are no interaction terms, since there is only one feature.

Build the polynomial features yourself using PyTorch. Do not use `PolynomialFeatures`.

- Standardize each polynomial column (subtract the mean, divide by the standard deviation) using statistics computed on the 100 training samples you fit on, then apply those same statistics to the test set.

Do not standardize the leading column of ones.

Use `torch.std(..., correction=0)` when calculating the standard deviations for this task.

- Use the same 80/20 split from Task 2.1, but fit on only the first 100 training samples (`X_train[:100]`, `y_train[:100]`).

Evaluate on the full test set.

Restricting the training set is what makes the variance of the high-degree models visible; with all 353 training points, the effect is muted.

**Required fitting method: Normal equations with numerical stabilization**

Compute the least-squares solution using the normal equations.

For numerical stability at high polynomial degrees, add $10^{-8}$ to the diagonal of $\widetilde{\mathbf{X}}_d^T\widetilde{\mathbf{X}}_d$ for all entries except the intercept entry before solving.

Specifically, calculate

$$
\boxed{
\mathbf{w}_d^{*}
=
\left(
\widetilde{\mathbf{X}}_d^T
\widetilde{\mathbf{X}}_d
+
10^{-8}\mathbf{P}_d
\right)^{-1}
\widetilde{\mathbf{X}}_d^T
\mathbf{y}_{\text{train}}
}
$$

where

$$
\mathbf{P}_d
=
\operatorname{diag}(0,1,1,\ldots,1).
$$

The first diagonal entry is zero because the first column of the polynomial data matrix is the intercept column.

Use PyTorch matrix operations and `torch.linalg.solve()` to solve the resulting linear system.

Do not calculate the matrix inverse explicitly.

The $10^{-8}$ diagonal term is included for numerical stability and must be used for every polynomial degree in this task.

Produce a single figure plotting training MSE and test MSE against polynomial degree, with both curves labelled.

Use a logarithmic y-axis, since the test MSE at the highest degrees is more than fifteen times larger than at the lowest.

Report:

- The degree at which test MSE is minimized and its value.
- The degree at which the gap between training and test MSE begins to grow sharply.

Then answer in two sentences:

Which range of degrees is the underfitting regime, and which is the overfitting regime?

In [ ]:
# Your code here

*Your answer here.*

---
## Question 5: Validation and Model Selection (20 points)


### Task 5.1 (12 points): K-Fold Cross-Validation, Implemented from Scratch

Write a function

```python
def kfold_indices(n, k, seed=42):
    """Return a list of k tuples (train_idx, val_idx) of integer index tensors."""
```

that shuffles the indices `0, ..., n-1` once using the given seed, splits them into `k` contiguous folds of as equal size as possible, and returns for each fold the validation indices and the complementary training indices.

Use PyTorch for shuffling and constructing the index tensors.

Use a local `torch.Generator` initialized with the supplied seed so that the function produces reproducible results.

You may not use `sklearn.model_selection.KFold`.

Using $k=5$ and the grid

`lambdas = torch.logspace(-3, 3, 30, dtype=torch.float64)`

perform the following:

- For each $\lambda$, run 5-fold cross-validation on the training set only and record the mean validation MSE for both Ridge and Lasso.
- For Ridge Regression, use the closed-form solution from Task 2.3.
- For Lasso, use the permitted `sklearn.linear_model.Lasso` implementation.
- Select the $\lambda$ with the lowest mean validation MSE for each model. If several values give the same mean validation MSE, select the smallest.
- Plot mean validation MSE against $\log_{10}(\lambda)$ for both models on one figure, marking the selected $\lambda$ for each.
- Refit each model on the full training set at its selected $\lambda$ and report the resulting test MSE.

As in Task 3.1, the Ridge and Lasso objectives normalize the squared-error term differently, so a given $\lambda$ does not denote the same penalty strength for both. Compare each model against itself across $\lambda$.

Report the selected $\lambda$ and final test MSE for Ridge and for Lasso.

In [ ]:
# Your code here


### Task 5.2 (8 points): What Goes Wrong If You Tune on the Test Set

Repeat the selection of $\lambda$ for Ridge over the same grid

`lambdas = torch.logspace(-3, 3, 30, dtype=torch.float64)`

but this time fit on the full training set at each $\lambda$ and pick the $\lambda$ that minimizes the test MSE directly.

Use the closed-form Ridge Regression solution from Task 2.3 for every value of $\lambda$.

Report:

- The $\lambda$ selected by directly minimizing test MSE and its test MSE.
- The $\lambda$ selected by cross-validation in Task 5.1 and its test MSE.
- The difference between the two test MSE values.

Then answer in **2–3 sentences**:

Both procedures produce a number you could call "the test error," so which one is an honest estimate of performance on unseen data, and what role does the validation set play that the test set cannot play once you have used it for selection?

In [ ]:
# Your code here

*Your answer here.*

---
### Submission

Submit this notebook as a single `.ipynb` file with **all cells executed and all outputs visible**